# Lesson 2.3 — Context Window Management: Drift, Bloat, and Compaction

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain context drift and context bloat and their causes
- Prune tool output and stale turns to control token growth
- Apply compaction (summarization) to long conversations
- Budget a context window across system, history, tools, and output

**You will produce:** A context budgeter plus a compaction function that summarizes old turns.

## Concepts

As conversations and agent loops grow, two failure modes appear:

- **Context bloat**: the window fills with stale turns, verbose tool outputs, and repeated data, raising cost/latency and pushing out useful content.
- **Context drift**: the model loses track of the original goal as irrelevant content accumulates.

Mitigations:
- **Tool-output pruning**: keep only the fields you need from large tool results.
- **Compaction**: periodically summarize older turns into a compact memory and drop the raw turns.
- **Budgeting**: allocate the window deliberately - reserve room for system, recent history, tool schemas, retrieved data, and the output `max_tokens`.

> **Reference:** [Anthropic - Context windows](https://docs.anthropic.com/en/docs/build-with-claude/context-windows)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Prune a large tool result

Keep only needed fields before adding tool output to the context.

In [ ]:
def prune_tool_result(raw, keep=("id", "status", "total")):
    return {k: raw.get(k) for k in keep if k in raw}

big = {"id": 7, "status": "ok", "total": 42, "debug": "x" * 5000, "trace": [1]*999}
print(prune_tool_result(big))

## Code-Along: Step 2 — Compact old turns

Summarize everything except the last few turns into one memory note.

In [ ]:
def compact(history, keep_recent=2):
    if len(history) <= keep_recent:
        return history
    old, recent = history[:-keep_recent], history[-keep_recent:]
    joined = " ".join(t["content"] for t in old if isinstance(t["content"], str))
    if HAVE_API:
        s = client.messages.create(model=MODEL_HAIKU, max_tokens=120,
            messages=[{"role": "user", "content": f"Summarize for memory: {joined}"}])
        summary = s.content[0].text
    else:
        summary = f"[MOCK summary of {len(old)} turns]"
    return [{"role": "user", "content": f"Conversation so far: {summary}"}] + recent

hist = [{"role": "user", "content": f"turn {i}"} for i in range(6)]
print(compact(hist))

## Exercise

Write `budget(window, system_tokens, tool_tokens, output_max)` that returns the token budget left for conversation history, or raises if the fixed parts already exceed the window.

In [ ]:
def budget(window, system_tokens, tool_tokens, output_max):
    # TODO: fixed = system_tokens + tool_tokens + output_max
    # TODO: if fixed >= window: raise ValueError
    # TODO: return window - fixed (room for history)
    pass

## Key Takeaways

- Context bloat wastes tokens; context drift loses the original goal.
- Prune verbose tool output to only the fields you need.
- Compact older turns into summaries and drop the raw history.
- Budget the window across system, history, tools, and output max_tokens.

---

**Next:** [Lesson 2.4 — Context Isolation with Subagents](./lesson-2.4-context-isolation-subagents.ipynb)